## Imports e configurações

In [21]:
# !pip install transformers[torch] torch torchmetrics pandas numpy scikit-learn pyarrow sentencepiece protobuf

In [22]:
USE_COLAB = False
USE_DRIVE = False

if USE_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [23]:
import warnings
warnings.filterwarnings("ignore")

import re
import json
import time
import math
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Optimizer

from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModel, get_scheduler

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, hamming_loss
)

# ─── Configurações globais ───────────────────────────────────────
RANDOM_STATE = 42

if USE_COLAB and USE_DRIVE:
    _DRIVE_ROOT = Path("/content/drive/MyDrive/Fuzzy_BERT")  # ← ajuste o nome da pasta
    DATA_DIR = _DRIVE_ROOT / "data/treated"
    OUT_DIR  = _DRIVE_ROOT / "data/out/albertina"
    CKPT_DIR = _DRIVE_ROOT / "data/checkpoints/albertina"
else:
    DATA_DIR = Path("../data/treated")
    OUT_DIR  = Path("../data/out/albertina")
    CKPT_DIR = Path("../data/checkpoints/albertina")

N_OUTER_FOLDS = 5
MODEL_NAME = "PORTULAN/albertina-100m-portuguese-ptbr-encoder" # Albertina 100M PT-BR (DeBERTa, 12L, hidden=768)
TEXT_COL = "CLEAN_TEXT"
MAX_LEN = 128
BATCH_SIZE = 32
NUM_EPOCHS = 6
WARMUP_RATIO = 0.1 # 10% dos steps para warmup linear
LEARNING_RATE = 2e-5
THRESHOLD = 0.5 # limiar para binarizar probabilidades

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMOTION_COLS = [
    'admiration', 'amusement', 'anger', 'annoyance',
    'approval', 'caring', 'confusion', 'curiosity', 'desire',
    'disappointment', 'disapproval', 'disgust', 'embarrassment',
    'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]
N_LABELS = len(EMOTION_COLS)

OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Device: {DEVICE}")
print(f"Modelo: {MODEL_NAME}")
print(f"Labels: {N_LABELS}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max length: {MAX_LEN}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Checkpoints em: {CKPT_DIR.resolve()}")

Device: cuda
Modelo: PORTULAN/albertina-100m-portuguese-ptbr-encoder
Labels: 28
Epochs: 6
Batch size: 32
Max length: 128
Learning rate: 2e-05
Checkpoints em: C:\Users\redga\OneDrive\Área de Trabalho\UFJF\02 - TCC\tcc-emotion-detection-nlp\data\checkpoints\albertina


## Carregamento dos folds

In [24]:
def load_fold(fold_id: int):
    """
    Carrega data.parquet e idx.npy de um fold.
    Retorna (df, idx_array).
    """
    fold_dir = DATA_DIR / f"fold_{fold_id}"
    df  = pd.read_parquet(fold_dir / "data.parquet")
    idx = np.load(fold_dir / "idx.npy")
    return df, idx

# Verifica disponibilidade dos folds
folds_data = {}
for k in range(1, N_OUTER_FOLDS + 1):
    df_k, idx_k = load_fold(k)
    folds_data[k] = {"df": df_k, "idx": idx_k}
    print(f"Fold {k}: {df_k.shape[0]:>6} amostras | idx shape: {idx_k.shape}")

# Inspeciona colunas do primeiro fold
print("\nColunas disponíveis:", folds_data[1]["df"].columns.tolist())

Fold 1:  10831 amostras | idx shape: (10831,)
Fold 2:  10810 amostras | idx shape: (10810,)
Fold 3:  10836 amostras | idx shape: (10836,)
Fold 4:  10909 amostras | idx shape: (10909,)
Fold 5:  10848 amostras | idx shape: (10848,)

Colunas disponíveis: ['id', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral', 'text', 'texto', 'UNCLEAN_TEXT', 'CLEAN_TEXT', 'letter_count', 'sentence_count', 'avg_letter_per_word', 'words_per_sentence', 'avg_syllables_per_word', 'reading_time', 'punctuation_per_word', 'exclamation_per_word', 'question_per_word', 'mean_verbs_per_word', 'mean_auxiliaries_per_word', 'mean_auxiliaries_per_sentence', 'flesch_portuguese', 'uppercase_percentage', 'mean_unique_word_length', 'mean_char_rep

## Dataset e DataLoader

In [25]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    """Dataset PyTorch para classificação multi-label de emoções."""

    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.texts = df[TEXT_COL].fillna("").astype(str).tolist()
        present = [c for c in EMOTION_COLS if c in df.columns]
        self.labels = df[present].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length = self.max_len,
            truncation = True,
        )
        encoding["labels"] = self.labels[idx].tolist()
        return encoding


def make_loaders(df_train, df_test):
    ds_train = EmotionDataset(df_train, tokenizer)
    ds_test  = EmotionDataset(df_test,  tokenizer)

    collator = DataCollatorWithPadding(tokenizer, padding="longest", return_tensors="pt")

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True, collate_fn=collator)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True, collate_fn=collator)
    return loader_train, loader_test

print("Tokenizer carregado:", MODEL_NAME)

Tokenizer carregado: PORTULAN/albertina-100m-portuguese-ptbr-encoder


## Modelo — Albertina + cabeça de classificação multi-label

In [26]:
class AlbertinaForEmotions(nn.Module):
    """
    Albertina 100M PT-BR (DeBERTa) com cabeça linear multi-label.
    Usa o [CLS] token (posição 0) como representação da sequência.
    """

    def __init__(self, model_name: str = MODEL_NAME, n_labels: int = N_LABELS,
                 dropout: float = 0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, n_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # [CLS]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)          # (B, N_LABELS)
        return logits

## Cross Balanced Loss (CB Loss)

Implementação da **Class-Balanced Loss** baseada em:
> Cui et al. (2019). *Class-Balanced Loss Based on Effective Number of Samples*. CVPR.

$$\text{CB}(p, y) = \frac{1 - \beta}{1 - \beta^{n_y}} \cdot \ell(p, y)$$

onde $n_y$ é o número de amostras da classe $y$ e $\beta = (N-1)/N$.

In [27]:
class CrossBalancedLoss(nn.Module):
    """
    Class-Balanced Binary Cross-Entropy para classificação multi-label.

    Parâmetros
    ----------
    samples_per_class : array-like (N_LABELS,)
        Número de amostras positivas por label no conjunto de treino.
    beta : float
        Hiperparâmetro de suavização (padrão 0.9999, original do paper).
        Valores comuns: 0.9, 0.99, 0.999, 0.9999.
    """

    def __init__(self, samples_per_class: np.ndarray, beta: float = 0.9999):
        super().__init__()

        # Número efetivo de amostras: EN = (1 - beta^n) / (1 - beta)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        # Peso inversamente proporcional ao número efetivo
        weights = (1.0 - beta) / np.where(effective_num == 0, 1e-8, effective_num)
        # Normaliza para que a soma dos pesos = N_LABELS
        weights = weights / weights.sum() * len(samples_per_class)

        self.register_buffer(
            "weights",
            torch.tensor(weights, dtype=torch.float32)
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, N_LABELS) — logits brutos do modelo
        targets : (B, N_LABELS) — rótulos binários {0, 1}
        """
        # BCE por elemento, sem redução
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )   # (B, N_LABELS)

        # Aplica os pesos de balanceamento por label
        weighted_bce = bce * self.weights.unsqueeze(0)  # broadcast em batch

        return weighted_bce.mean()

def compute_samples_per_class(df: pd.DataFrame) -> np.ndarray:
    """Conta positivos por label. Labels ausentes recebem 1 (evita divisão por zero)."""
    counts = np.array([
        int(df[c].sum()) if c in df.columns else 1
        for c in EMOTION_COLS
    ], dtype=np.float64)
    return np.maximum(counts, 1)   # garante >= 1

## Otimizador e Scheduler

AdamW nativo do PyTorch com weight decay desacoplado.
Exclui LayerNorm e bias do weight decay, conforme recomendado para fine-tuning BERT/DeBERTa.

In [28]:
def build_optimizer(model: nn.Module, lr: float = LEARNING_RATE) -> torch.optim.AdamW:
    """
    AdamW nativo do PyTorch com grupos de parâmetros:
      - com weight decay : todos exceto LayerNorm, layer_norm e bias
      - sem weight decay : LayerNorm, layer_norm, bias
    Equivalente ao AdamWeightDecayOptimizer original do BERT,
    mas implementado em C++/CUDA — ordens de magnitude mais rápido.
    """
    exclude_patterns = ["LayerNorm", "layer_norm", "bias"]

    decay_params    = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(re.search(pat, name) for pat in exclude_patterns):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    param_groups = [
        {"params": decay_params,    "weight_decay": 0.01},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    return torch.optim.AdamW(
        param_groups,
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-6,
    )


def build_polynomial_scheduler(optimizer, num_train_steps: int, warmup_ratio: float = WARMUP_RATIO):
    """
    Polynomial decay com warmup linear.
    Fase 1 (warmup): lr cresce linearmente de 0 até lr_inicial
    Fase 2 (decay) : lr decai linearmente (power=1.0) até 0
    """
    num_warmup_steps = int(num_train_steps * warmup_ratio)

    def lr_lambda(current_step: int):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_train_steps - num_warmup_steps)
        )
        return max(0.0, 1.0 - progress)

    from torch.optim.lr_scheduler import LambdaLR
    return LambdaLR(optimizer, lr_lambda)

## Funções de treino, avaliação e checkpoint

In [29]:
def save_checkpoint(model, optimizer, scheduler, epoch, fold, metrics, ckpt_dir: Path):
    """
    Salva um checkpoint ao final de cada época.
    Arquivo: ckpt_dir/fold_{fold}_epoch_{epoch}.pt
    """
    ckpt_path = ckpt_dir / f"fold_{fold}_epoch_{epoch:02d}.pt"
    torch.save({
        "epoch": epoch,
        "fold": fold,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": metrics,
    }, ckpt_path)
    print(f"✓ Checkpoint salvo: {ckpt_path.name}")
    return ckpt_path

def load_checkpoint(ckpt_path: Path, model, optimizer=None, scheduler=None):
    import numpy.dtypes
    torch.serialization.add_safe_globals([
        numpy.dtypes.Float64DType,
    ])
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    print(f"Checkpoint carregado: fold={ckpt['fold']}, epoch={ckpt['epoch']}")
    return ckpt

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    """Treina o modelo por uma época. Retorna loss médio."""
    model.train()
    total_loss = 0.0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()

        # Gradient clipping (recomendado para fine-tuning BERT)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold: float = THRESHOLD):
    """Avalia o modelo. Retorna loss, Y_true, Y_pred, Y_proba."""
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        total_loss += loss.item()
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits, dim=0)   # (N, N_LABELS)
    all_labels = torch.cat(all_labels, dim=0)

    Y_proba = torch.sigmoid(all_logits).numpy()
    Y_pred = (Y_proba >= threshold).astype(int)
    Y_true = all_labels.numpy().astype(int)

    avg_loss = total_loss / len(loader)
    return avg_loss, Y_true, Y_pred, Y_proba

def compute_metrics(Y_true, Y_pred, Y_proba, label_names):
    """Calcula métricas multi-label idênticas ao baseline."""
    metrics = {
        "accuracy": accuracy_score(Y_true, Y_pred),
        "hamming_loss": hamming_loss(Y_true, Y_pred),
        "f1_macro": f1_score(Y_true, Y_pred, average="macro",    zero_division=0),
        "f1_micro": f1_score(Y_true, Y_pred, average="micro",    zero_division=0),
        "f1_weighted": f1_score(Y_true, Y_pred, average="weighted", zero_division=0),
        "precision_macro": precision_score(Y_true, Y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(Y_true, Y_pred, average="macro",    zero_division=0),
    }

    valid_cols = [j for j in range(Y_true.shape[1]) if Y_true[:, j].sum() > 0]
    if valid_cols:
        metrics["roc_auc_macro"] = roc_auc_score(
            Y_true[:, valid_cols], Y_proba[:, valid_cols], average="macro"
        )
    else:
        metrics["roc_auc_macro"] = float("nan")

    f1_per = f1_score(Y_true, Y_pred, average=None, zero_division=0)
    metrics["f1_per_label"] = dict(zip(label_names, f1_per))

    return metrics

## Loop principa — Outer CV com Albertina

In [30]:
all_fold_metrics = []

for outer_k in range(1, N_OUTER_FOLDS + 1):
    t0 = time.time()
    print(f"\n{'='*65}")
    print(f"  OUTER FOLD {outer_k}/{N_OUTER_FOLDS}")
    print(f"{'='*65}")

    # ── Dados ──────────────────────────────────────────────────────
    df_test_outer, _ = load_fold(outer_k)

    train_dfs = [
        folds_data[k]["df"]
        for k in range(1, N_OUTER_FOLDS + 1)
        if k != outer_k
    ]
    df_train_outer = pd.concat(train_dfs, ignore_index=True)

    print(f"  Treino : {len(df_train_outer):>6} | Teste: {len(df_test_outer):>5}")

    loader_train, loader_test = make_loaders(df_train_outer, df_test_outer)

    # ── Cross Balanced Loss ────────────────────────────────────────
    samples_per_class = compute_samples_per_class(df_train_outer)
    criterion = CrossBalancedLoss(samples_per_class, beta=0.9999).to(DEVICE)
    print(f"  CB Loss configurada. Positivos por label (min/max): "
          f"{samples_per_class.min():.0f} / {samples_per_class.max():.0f}")

    # ── Modelo ─────────────────────────────────────────────────────
    model = AlbertinaForEmotions().to(DEVICE)
    print(torch.cuda.memory_allocated() / 1e9, "GB alocados na GPU")

    # ── Otimizador e scheduler ─────────────────────────────────────
    num_train_steps = len(loader_train) * NUM_EPOCHS
    optimizer = build_optimizer(model, lr=LEARNING_RATE)
    scheduler = build_polynomial_scheduler(optimizer, num_train_steps, WARMUP_RATIO)

    print(f"  Steps totais   : {num_train_steps}")
    print(f"  Warmup steps   : {int(num_train_steps * WARMUP_RATIO)}")

    # ── Verifica se há checkpoint do fold para retomar ─────────────
    start_epoch = 1
    existing_ckpts = sorted(CKPT_DIR.glob(f"fold_{outer_k}_epoch_*.pt"))
    if existing_ckpts:
        latest_ckpt = existing_ckpts[-1]
        print(f"  Retomando do checkpoint: {latest_ckpt.name}")
        ckpt = load_checkpoint(latest_ckpt, model, optimizer, scheduler)
        start_epoch = ckpt["epoch"] + 1

        # ── Fold já completo: recupera métricas do checkpoint ─────
        if start_epoch > NUM_EPOCHS:
            print(f"  Fold {outer_k} já completo. Recuperando métricas do checkpoint.")
            best_ep = ckpt["metrics"]
            best_ep["elapsed_s"] = round(time.time() - t0, 1)
            all_fold_metrics.append(best_ep)
            continue

    # ── Loop de épocas ────────────────────────────────────────────
    epoch_metrics_history = []

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        t_ep = time.time()
        print(f"\n  ── Época {epoch}/{NUM_EPOCHS} ──")

        # Treino
        train_loss = train_one_epoch(
            model, loader_train, optimizer, scheduler, criterion, DEVICE
        )

        # Avaliação no conjunto de teste do fold
        val_loss, Y_true, Y_pred, Y_proba = evaluate(
            model, loader_test, criterion, DEVICE, threshold=THRESHOLD
        )

        epoch_met = compute_metrics(Y_true, Y_pred, Y_proba, EMOTION_COLS)
        epoch_met["train_loss"] = round(train_loss, 4)
        epoch_met["val_loss"] = round(val_loss, 4)
        epoch_met["epoch"] = epoch
        epoch_met["fold"] = outer_k
        epoch_metrics_history.append(epoch_met)

        elapsed_ep = round(time.time() - t_ep, 1)
        print(f"    Train loss : {train_loss:.4f}")
        print(f"    Val loss   : {val_loss:.4f}")
        print(f"    F1-Macro   : {epoch_met['f1_macro']:.4f}")
        print(f"    F1-Micro   : {epoch_met['f1_micro']:.4f}")
        print(f"    ROC-AUC    : {epoch_met['roc_auc_macro']:.4f}")
        print(f"    Tempo ep.  : {elapsed_ep}s")

        # ── Checkpoint ao final de cada época ─────────────────────
        save_checkpoint(
            model, optimizer, scheduler,
            epoch=epoch, fold=outer_k,
            metrics=epoch_met,
            ckpt_dir=CKPT_DIR
        )

    # ── Melhor época deste fold (maior F1-Macro no teste) ─────────
    best_ep = max(epoch_metrics_history, key=lambda m: m["f1_macro"])
    best_ep["elapsed_s"] = round(time.time() - t0, 1)
    all_fold_metrics.append(best_ep)

    print(f"\n  Melhor época do fold {outer_k}: {best_ep['epoch']} "
          f"| F1-Macro={best_ep['f1_macro']:.4f}")

print("\n✓ Outer CV concluído.")


  OUTER FOLD 1/5
  Treino :  43403 | Teste: 10831
  CB Loss configurada. Positivos por label (min/max): 77 / 14199


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 2771.18it/s]
[transformers] DebertaModel LOAD REPORT from: PORTULAN/albertina-100m-portuguese-ptbr-encoder
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1.821390848 GB alocados na GPU
  Steps totais   : 8142
  Warmup steps   : 814

  ── Época 1/6 ──


KeyboardInterrupt: 

## Análise dos resultadso

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

global_metric_keys = [
    "fold", "epoch", "accuracy", "hamming_loss",
    "f1_macro", "f1_micro", "f1_weighted",
    "precision_macro", "recall_macro", "roc_auc_macro",
    "train_loss", "val_loss", "elapsed_s"
]

rows = [{k: m.get(k) for k in global_metric_keys} for m in all_fold_metrics]
df_results = pd.DataFrame(rows).set_index("fold")

numeric_cols = [c for c in df_results.columns if c not in ("elapsed_s", "epoch")]
mean_row = df_results[numeric_cols].mean().rename("mean")
std_row  = df_results[numeric_cols].std().rename("std")

df_summary = pd.concat([
    df_results[numeric_cols],
    mean_row.to_frame().T,
    std_row.to_frame().T
])

print("\n===== MÉTRICAS GLOBAIS POR OUTER FOLD (melhor época) =====")
display(df_results.round(4))

print("\n===== RESUMO (média ± desvio-padrão) =====")
display(df_summary.round(4))

In [ ]:
# F1 por label (média entre folds)
f1_per_label_folds = pd.DataFrame(
    [m["f1_per_label"] for m in all_fold_metrics],
    index=[f"fold_{m['fold']}" for m in all_fold_metrics]
)

f1_per_label_mean = f1_per_label_folds.mean().sort_values(ascending=False)

print("\n===== F1 POR LABEL (média entre folds) =====")
display(pd.DataFrame({
    "F1 Médio" : f1_per_label_mean,
    "Std"      : f1_per_label_folds.std()
}).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_plot = ["f1_macro", "f1_micro", "f1_weighted", "roc_auc_macro"]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

ax = axes[0]
x = np.arange(len(df_results))
width = 0.20
for i, (met, col) in enumerate(zip(metric_plot, colors)):
    ax.bar(x + i * width, df_results[met], width, label=met, color=col, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([f"Fold {k}" for k in df_results.index])
ax.set_ylim(0, 1)
ax.set_title("Albertina — Métricas por Outer Fold (melhor época)")
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.set_ylabel("Score")

ax2 = axes[1]
f1_per_label_mean.plot.barh(ax=ax2, color="#4C72B0", alpha=0.85)
ax2.set_title("F1 por Label (média entre folds)")
ax2.set_xlabel("F1 Score")
ax2.set_xlim(0, 1)
ax2.axvline(
    f1_per_label_mean.mean(), color="red", linestyle="--",
    label=f"Média={f1_per_label_mean.mean():.3f}"
)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "albertina_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura salva em {OUT_DIR / 'albertina_metrics.png'}")

In [ ]:
# Métricas globais
df_results.to_csv(OUT_DIR / "albertina_global_metrics.csv")

# F1 por label
f1_per_label_folds.to_csv(OUT_DIR / "albertina_f1_per_label.csv")

# Resumo JSON
summary_dict = {
    "model": "Albertina 100M PT-BR (PORTULAN/albertina-100m-portuguese-ptbr-encoder)",
    "loss": "CrossBalancedLoss (beta=0.9999)",
    "optimizer": "AdamWeightDecay (lr=2e-5, wd=0.01)",
    "scheduler": "Polynomial decay + linear warmup (10%)",
    "strategy": f"Outer CV ({N_OUTER_FOLDS} folds), {NUM_EPOCHS} épocas",
    "n_labels": N_LABELS,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "f1_macro_mean": round(float(df_results["f1_macro"].mean()), 4),
    "f1_macro_std": round(float(df_results["f1_macro"].std()),  4),
    "f1_micro_mean": round(float(df_results["f1_micro"].mean()), 4),
    "f1_micro_std": round(float(df_results["f1_micro"].std()),  4),
    "roc_auc_mean": round(float(df_results["roc_auc_macro"].mean()), 4),
    "roc_auc_std": round(float(df_results["roc_auc_macro"].std()),  4),
}

with open(OUT_DIR / "albertina_summary.json", "w") as f:
    json.dump(summary_dict, f, indent=2, ensure_ascii=False)

print("Arquivos exportados para:", OUT_DIR.resolve())
print(json.dumps(summary_dict, indent=2, ensure_ascii=False))